<a href="https://colab.research.google.com/github/lygitdata/GarmentIQ/blob/main/test/adv_usage_custom_measurement_instruction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Advanced Usage - GarmentIQ Custom Measurement Instruction

A measurement instruction tells GarmentIQ which landmarks a garment has and which pairs of
them form a measurement. The defaults cover common garments, but you may want different
measurements, or a garment of your own.

This tutorial shows how to read the predefined instructions, write a custom one, register
it in a garment class dictionary, and confirm that landmark detection returns exactly the
measurements you asked for.

## Table of Contents

1. [Prerequisites](#prerequisites)
2. [Inspect the predefined instructions](#inspect)
3. [Write a custom instruction](#write)
4. [Register the instruction](#register)
5. [Compare the results](#compare)

<a name="prerequisites"></a>
## Prerequisites

Install the package and download a skirt image and the landmark detection model. On Colab
you can keep this section collapsed.

In [ ]:
# @title Install GarmentIQ
!pip install garmentiq -q

In [ ]:
# @title Import GarmentIQ and choose a device

import copy
import json

import torch

import garmentiq as giq
from garmentiq.garment_classes import garment_classes
from garmentiq.landmark.detection.model_definition import PoseHighResolutionNet

# GarmentIQ never grabs an accelerator on its own: every model loader and every
# inference function takes a `device` argument that defaults to "cpu". Pass it
# explicitly to use a GPU ("cuda") or Apple Silicon ("mps").
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Using device:", device)

In [ ]:
# @title Download the test image and the landmark detection model

!mkdir -p ./test_image
!wget -q -O ./test_image/cloth_4.jpg \
    https://raw.githubusercontent.com/lygitdata/GarmentIQ/refs/heads/gh-pages/asset/img/cloth_4.jpg

!mkdir -p ./models
!wget -q -O ./models/hrnet.pth \
    https://huggingface.co/lygitdata/garmentiq/resolve/main/hrnet.pth

print("Downloads finished.")

<a name="inspect"></a>
## Inspect the predefined instructions

`garment_classes` maps each garment to its metadata. Three fields matter here:

| Field | Meaning |
|---|---|
| `num_predefined_points` | how many landmarks the detection model predicts |
| `index_range` | which slice of the model's output belongs to this class |
| `instruction` | path to the JSON file defining landmarks and measurements |

`num_predefined_points` and `index_range` are fixed by the training data (DeepFashion2),
so change them only if you retrain the detection model. The `instruction` file is yours to
replace, and that is what this tutorial does.

In [ ]:
print(json.dumps(garment_classes, indent=4))

By default a skirt has three measurements: `waist`, `full length`, and
`hips`. Suppose you only want `waist` and `hips`.

In [ ]:
with open(garment_classes["skirt"]["instruction"]) as fh:
    default_skirt = json.load(fh)

print(json.dumps(default_skirt, indent=4))

<a name="write"></a>
## Write a custom instruction

An instruction has two parts. `landmarks` lists the points, each keyed by its landmark ID,
and `measurements` names each measurement by the pair of landmark IDs it spans.

The IDs must match the ones the detection model predicts for that class, which is why the
`x` and `y` values below are only placeholders: detection overwrites them at run time.
Points marked `predefined: True` come from the model, while `predefined: False` points are
computed by the derivation step.

> You can generate instructions visually with the
> [GarmentIQ instruction generation tool](https://garmentiq.ly.gd.edu.kg/application/demo/instruction-generation/)
> instead of writing the JSON by hand.

In [ ]:
skirt_new = {
    "skirt": {
        "landmarks": {
            "1": {
                "predefined": True,
                "description": "waist_left",
                "x": 60,
                "y": 40,
            },
            "3": {
                "predefined": True,
                "description": "waist_right",
                "x": 140,
                "y": 40,
            },
            "4": {
                "predefined": True,
                "description": "side_seam_left",
                "x": 50,
                "y": 80,
            },
            "8": {
                "predefined": True,
                "description": "side_seam_right",
                "x": 150,
                "y": 80,
            },
        },
        "measurements": {
            "waist": {
                "landmarks": {"start": "1", "end": "3"},
                "description": "/",
            },
            "hips": {
                "landmarks": {"start": "4", "end": "8"},
                "description": "/",
            },
        },
    }
}

with open("skirt_new.json", "w") as fh:
    json.dump(skirt_new, fh, indent=4)

print("Wrote skirt_new.json")

<a name="register"></a>
## Register the instruction

Point a garment class at the new file. Copy the dictionary rather than editing
`garment_classes` in place, so the defaults stay available for comparison.

In [ ]:
new_garment_classes = copy.deepcopy(garment_classes)
new_garment_classes["skirt"]["instruction"] = "skirt_new.json"

print(new_garment_classes["skirt"])

<a name="compare"></a>
## Compare the results

Run detection twice on the same image, changing only `class_dict`. Everything else, the
model and all its arguments, stays identical.

In [ ]:
giq.landmark.plot(image_path="./test_image/cloth_4.jpg", figsize=(3, 3))

HRNet = giq.landmark.detection.load_model(
    model_path="./models/hrnet.pth",
    model_class=PoseHighResolutionNet(),
    device=device,
)

In [ ]:
# The default instruction
_, _, detection_dict = giq.landmark.detect(
    class_name="skirt",
    class_dict=garment_classes,
    image_path="./test_image/cloth_4.jpg",
    model=HRNet,
    scale_std=200.0,
    resize_dim=[288, 384],
    normalize_mean=[0.485, 0.456, 0.406],
    normalize_std=[0.229, 0.224, 0.225],
    device=device,
)

# `clean_detection_dict` trims the record down to the measurements themselves
default_result = giq.utils.clean_detection_dict(
    class_name="skirt",
    image_name="cloth_4.jpg",
    detection_dict=detection_dict,
)

print("Default measurements:", list(default_result["cloth_4.jpg"]["measurements"]))
default_result

In [ ]:
# The custom instruction. Only `class_dict` changes.
_, _, detection_dict_new = giq.landmark.detect(
    class_name="skirt",
    class_dict=new_garment_classes,
    image_path="./test_image/cloth_4.jpg",
    model=HRNet,
    scale_std=200.0,
    resize_dim=[288, 384],
    normalize_mean=[0.485, 0.456, 0.406],
    normalize_std=[0.229, 0.224, 0.225],
    device=device,
)

custom_result = giq.utils.clean_detection_dict(
    class_name="skirt",
    image_name="cloth_4.jpg",
    detection_dict=detection_dict_new,
)

print("Custom measurements:", list(custom_result["cloth_4.jpg"]["measurements"]))
custom_result

The custom instruction returns only `waist` and `hips`, as intended.

`new_garment_classes` can be passed anywhere a class dictionary is accepted, including the
`class_dict` argument of the `tailor` pipeline, so a custom instruction flows through to
the final measurements. See the
[tailor tutorial](https://colab.research.google.com/github/lygitdata/GarmentIQ/blob/main/test/tutorial_tailor.ipynb).